In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from scipy.stats import pearsonr

# === CONFIGURATION ===
proportions_file = 'TOO-Decon-Degradation/Data/20250616_All-Tissues-NoDup_Random_Simulated_v2_Proportions.txt'
root_dir = 'TOO-Decon-Degradation/'
results_dir = 'Results-Pearson-PredictedVSActual'
os.makedirs(results_dir, exist_ok=True)

# Use this method list for detection
method_map = ['BayesPrism', 'nuSVR', 'ReDeconv', 'CIBERSORTx', 'MuSiC', 'NNLS', 'QP']

# === LOAD PROPORTIONS FILE ===
prop_df = pd.read_csv(proportions_file, sep='\t', index_col=0)
prop_df['FemaleReproductive'] = prop_df[['Cervix', 'Ovary', 'Uterus']].sum(axis=1)
prop_df = prop_df.rename(columns={
    'Thyroid-gland': 'Thyroid',
    'Adrenal-gland': 'Adrenal gland',
    'Small-Intestine': 'SmallIntestine',
    'Skeletal-muscle': 'MuscleSkeletal'
})
prop_df = prop_df.drop(columns=['Cervix', 'Ovary', 'Uterus', 'Thyroid-gland', 'Adrenal-gland', 'Small-Intestine', 'Skeletal-muscle'], errors='ignore')

reference_tissues = [
    'Adipose', 'Adrenal gland', 'Arteries', 'Bladder', 'Brain', 'Breast', 'Colon', 'Esophagus', 'FemaleReproductive',
    'Fibroblasts', 'Heart', 'Kidney', 'Liver', 'Lung', 'Lymphocytes', 'MuscleSkeletal', 'NerveTibial', 'Pancreas',
    'Pituitary', 'Prostate', 'SalivaryGland', 'Skin', 'SmallIntestine', 'Spleen', 'Stomach', 'Testis', 'Thyroid', 'Whole blood'
]

# Add missing tissue columns and normalize
for tissue in reference_tissues:
    if tissue not in prop_df.columns:
        prop_df[tissue] = 0.0

prop_df_tissues = prop_df[reference_tissues].copy()
prop_df_tissues = prop_df_tissues.div(prop_df_tissues.sum(axis=1), axis=0).multiply(100).round(5)
prop_df.update(prop_df_tissues)

# === PROCESS EACH DECON FILE ===
decon_files = sorted(glob(os.path.join(root_dir, 'Decon-Results_*removed_2Median_300_500', '*_modified.txt')))

# Collect pearson results for tissues
results = []

for file_path in decon_files:
    try:
        dir_name = os.path.basename(os.path.dirname(file_path))

        # Extract matrix info (everything after last underscore chunk)
        matrix = dir_name.split('_')[-1]  # e.g. "2Median_300_500"

        # Extract gene removal info (look for "top_xremoved")
        gene_removal = "NA"
        parts = dir_name.split('_')
        for i, p in enumerate(parts):
            if p == "top" and i + 1 < len(parts):
                gene_removal = f"top_{parts[i+1]}"
                break

        filename = os.path.basename(file_path)
        raw_method = filename.replace('_modified.txt', '')
        method = next((m for m in method_map if m in raw_method), raw_method)

        plot_name_base = f'{matrix}_Random-TOO-V2_{method}_{gene_removal}'
        plot_png = os.path.join(results_dir, plot_name_base + ".png")
        plot_pdf = os.path.join(results_dir, plot_name_base + ".pdf")

        # Load decon data
        decon_df = pd.read_csv(file_path, sep='\t', index_col=0)
        for col in reference_tissues:
            if col not in decon_df.columns:
                decon_df[col] = 0.0
        decon_df = decon_df[reference_tissues]

        # Align samples
        common_samples = prop_df.index.intersection(decon_df.index)
        true_vals_df = prop_df.loc[common_samples, reference_tissues]
        pred_vals_df = decon_df.loc[common_samples, reference_tissues]

        # Flatten and remove zero-zero pairs
        true_vals_flat = true_vals_df.values.flatten()
        pred_vals_flat = pred_vals_df.values.flatten()
        mask_nonzero = ~((true_vals_flat == 0) & (pred_vals_flat == 0))
        true_vals_flat = true_vals_flat[mask_nonzero]
        pred_vals_flat = pred_vals_flat[mask_nonzero]

        # Pearson correlation
        r, p = pearsonr(true_vals_flat, pred_vals_flat)
        print(f"{plot_name_base} — Pearson r = {r:.4f}, p = {p:.2e}")
        results.append([matrix, method, gene_removal, r, r**2, p])

        
        # Plot
        plt.figure(figsize=(8, 8))
        sns.set(style="white", context="talk")
        ax = sns.regplot(
            x=true_vals_flat, 
            y=pred_vals_flat, 
            scatter_kws={'s': 30, 'alpha': 0.5, 'color': '#4C72B0'},
            line_kws={'color': 'crimson', 'lw': 2}
        )
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.set_xlabel('Actual Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_ylabel('Predicted Fraction (%)', fontsize=14, fontweight='bold')
        ax.set_title(f'{matrix} — {method}\nPearson r = {r:.2f}, p = {p:.2e}', fontsize=16, fontweight='bold')
        ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
        sns.despine()
        plt.tight_layout()
        plt.savefig(plot_png, dpi=300, bbox_inches="tight")
        plt.savefig(plot_pdf, dpi=300, bbox_inches="tight")
        plt.close()
        print(f"Saved plot: {plot_png} and {plot_pdf}")

        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Step 2: Save summary table
if results:
    results_df = pd.DataFrame(
    results, 
    columns=["Matrix", "Method", "GeneRemoval", "Pearson_r", "R_squared", "p_value"]
)
    results_csv = os.path.join(results_dir, "Random_Pearson_Summary.csv")
    results_df.to_csv(results_csv, index=False)
    print(f"Saved Pearson summary table: {results_csv}")
else:
    print("No results to save.")


500_Random-TOO-V2_BayesPrism_top_0 — Pearson r = 0.6076, p = 0.00e+00
Saved plot: Results-Pearson-PredictedVSActual/500_Random-TOO-V2_BayesPrism_top_0.png and Results-Pearson-PredictedVSActual/500_Random-TOO-V2_BayesPrism_top_0.pdf
500_Random-TOO-V2_MuSiC_top_0 — Pearson r = 0.2001, p = 3.19e-158
Saved plot: Results-Pearson-PredictedVSActual/500_Random-TOO-V2_MuSiC_top_0.png and Results-Pearson-PredictedVSActual/500_Random-TOO-V2_MuSiC_top_0.pdf
500_Random-TOO-V2_ReDeconv_top_0 — Pearson r = 0.6500, p = 0.00e+00
Saved plot: Results-Pearson-PredictedVSActual/500_Random-TOO-V2_ReDeconv_top_0.png and Results-Pearson-PredictedVSActual/500_Random-TOO-V2_ReDeconv_top_0.pdf
500_Random-TOO-V2_CIBERSORTx_top_0 — Pearson r = 0.4245, p = 0.00e+00
Saved plot: Results-Pearson-PredictedVSActual/500_Random-TOO-V2_CIBERSORTx_top_0.png and Results-Pearson-PredictedVSActual/500_Random-TOO-V2_CIBERSORTx_top_0.pdf
500_Random-TOO-V2_NNLS_top_0 — Pearson r = 0.2218, p = 6.09e-134
Saved plot: Results-Pearson

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none'
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.sans-serif'] = ['DejaVu Sans']

# === LOAD DATA ===
file_path = 'Results-Pearson-PredictedVSActual/Random_Pearson_Summary.csv'
df = pd.read_csv(file_path)

# === EXTRACT NUMERIC GENE REMOVAL VALUE FOR ORDERING ===
df['GeneRemovalNumeric'] = (
    df['GeneRemoval']
    .str.extract(r'top_(\d+)')
    .astype(float)
)

# === PIVOT TO METHOD × GENE REMOVAL ===
pivot_df = df.pivot_table(
    index='Method',
    columns='GeneRemovalNumeric',
    values='Pearson_r',   
    aggfunc='mean'
).sort_index(axis=1)

# === REORDER ROWS ===
method_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
pivot_df = pivot_df.reindex(method_order)

# === PLOT HEATMAP ===
fig, ax = plt.subplots(figsize=(6, 4), dpi=600, constrained_layout=True)
ax = sns.heatmap(
    pivot_df,
    cmap='coolwarm',
#    annot=True,
#    fmt=".1f",
#    annot_kws={"size": 10},
    cbar_kws={'shrink': 0.9},
    linewidths=0,
    linecolor='white',
    center=0 
)

# Force ticks to show
ax.tick_params(
    axis='x', which='both', bottom=True, top=False, labelbottom=True,
    length=4, width=0.6
)
ax.tick_params(
    axis='y', which='both', left=True, right=False, labelleft=True,
    length=4, width=0.6
)

# Colorbar styling
cbar = ax.collections[0].colorbar
cbar.ax.set_ylabel("Pearson r", fontsize=13, rotation=270, labelpad=20)   
cbar.ax.tick_params(labelsize=11, width=0.8, length=4)             
cbar.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.2f}"))

# === Axes styling ===
plt.ylabel('Deconvolution Tool', fontsize=14, labelpad=10)
plt.xlabel('Genes Removed (%)', fontsize=14, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)

# === SAVE FIGURE ===
fig.savefig("Pearson_Heatmap_Best-Matrix_Degradation_Clean.svg")
plt.close(fig)